In [26]:
import warnings
import pathlib

from PIL import Image

from torchvision import models
import torch
from torch import nn
import torch.functional as F

In [34]:
model = models.mobilenet_v2(pretrained=True)
model.classifier = nn.Linear(model.last_channel, 2)

model.load_state_dict(torch.load(r"mobilenet_v2_weights.pth"))

def convert_mobilenetv2_finn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d) and child.groups == child.in_channels:
            new_conv = nn.Conv2d(
                in_channels=child.in_channels,
                out_channels=child.out_channels,
                kernel_size=child.kernel_size,
                stride=child.stride,
                padding=child.padding,
                bias=(child.bias is not None)
            )
            new_conv.weight.data.copy_(child.weight.data)
            if child.bias is not None:
                new_conv.bias.data.copy_(child.bias.data)
            setattr(module, name, new_conv)

        elif isinstance(child, nn.ReLU6):
            setattr(module, name, nn.ReLU(inplace=True))

        else:
            convert_mobilenetv2_finn(child)


convert_mobilenetv2_finn(model)

In [37]:
torch.save(model, r"mobilenet_v2_conv2dBlock.pth")